# Croatian Word Lists – Combine & Validate

Build strict and non-strict 4-letter and 5-letter Croatian word lists for Word Ladder.

In [1]:
import sys
print("Python:", sys.version)

from pathlib import Path
print("Notebook ready.")

Python: 3.12.3 (tags/v3.12.3:f6650f9, Apr  9 2024, 14:05:25) [MSC v.1938 64 bit (AMD64)]
Notebook ready.


## Inspect output counts (like English notebook)

Quick cell to see how many words are in each Croatian dataset.

In [7]:
from pathlib import Path

files = [
    Path("../data/croatian_4.txt"),
    Path("../data/croatian_5.txt"),
    Path("../data/croatian_4_strict.txt"),
    Path("../data/croatian_5_strict.txt"),
]

for path in files:
    if not path.exists():
        print(f"Missing: {path}")
        continue

    words = [line.strip() for line in path.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
    unique_words = set(words)

    print(f"{path.name} -> total lines: {len(words)}, unique words: {len(unique_words)}")
    print(f"first 10: {words[:10]}")
    print()

croatian_4.txt -> total lines: 6309, unique words: 6309
first 10: ['abel', 'adam', 'aden', 'adut', 'afin', 'afta', 'afte', 'agar', 'agio', 'agip']

croatian_5.txt -> total lines: 19135, unique words: 19135
first 10: ['abaka', 'abake', 'abces', 'abeba', 'aceni', 'adaks', 'adama', 'adame', 'adams', 'adamu']

croatian_4_strict.txt -> total lines: 4099, unique words: 4099
first 10: ['adut', 'agip', 'agom', 'ajde', 'ajme', 'akna', 'akne', 'akni', 'aknu', 'aksa']

croatian_5_strict.txt -> total lines: 12228, unique words: 12228
first 10: ['adama', 'adame', 'adamu', 'aduta', 'adute', 'aduti', 'adutu', 'afekt', 'afera', 'afere']



## Download sources & build datasets

**Sources:**
- **Rijecalica** – game-cleaned wordlist (ISO-8859-2)
- **kkrypt0nn/wordlists** – large list
- **HR_Txt** – gigaly dictionary (624k+), already in `data/raw/`
- **Strict** = all 3 sources | **Non-strict** = >=2 sources

In [8]:
from pathlib import Path
from collections import Counter
from urllib.request import urlopen

base = Path("../data")
raw_dir = base / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

# Final outputs
out_5 = base / "croatian_5.txt"
out_5_strict = base / "croatian_5_strict.txt"
out_4 = base / "croatian_4.txt"
out_4_strict = base / "croatian_4_strict.txt"

# External sources
rijecalica_url = "https://raw.githubusercontent.com/Martinsos/Rijecalica/master/croatian-wordlist-checked-iso8859-2.txt"
rijecalica_file = raw_dir / "rijecalica_croatian.txt"

kkrypt0nn_url = "https://raw.githubusercontent.com/kkrypt0nn/wordlists/main/wordlists/languages/croatian.txt"
kkrypt0nn_file = raw_dir / "kkrypt0nn_croatian.txt"

hr_txt_file = raw_dir / "HR_Txt-624.txt"  # user-provided

# kkrypt0nn: UTF-8
if not kkrypt0nn_file.exists():
    print("Downloading kkrypt0nn_croatian.txt...")
    kkrypt0nn_file.write_text(
        urlopen(kkrypt0nn_url, timeout=60).read().decode("utf-8", errors="ignore"),
        encoding="utf-8",
    )

# Rijecalica: ISO-8859-2 (Central European)
if not rijecalica_file.exists():
    print("Downloading rijecalica_croatian.txt (ISO-8859-2)...")
    try:
        data = urlopen(rijecalica_url, timeout=60).read()
        text = data.decode("iso-8859-2", errors="ignore")
        rijecalica_file.write_text(text, encoding="utf-8")
    except Exception as e:
        print(f"Rijecalica download failed: {e}")


def load_set_simple(path: Path, n: int, encoding: str = "utf-8") -> set[str]:
    """Load words of length n. One word per line, or tab-separated (first col)."""
    if not path.exists():
        return set()
    words = set()
    for line in path.read_text(encoding=encoding, errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue
        # Handle tab-separated (HR_Txt: word\tlemma\t...)
        token = line.split("\t")[0].strip().lower()
        if len(token) == n and token.isalpha():
            words.add(token)
    return words


def load_rijecalica(path: Path, n: int) -> set[str]:
    """Rijecalica: we save as UTF-8 after download; try both encodings."""
    if not path.exists():
        return set()
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        text = path.read_text(encoding="iso-8859-2", errors="ignore")
    words = set()
    for line in text.splitlines():
        token = line.strip().lower()
        if token and len(token) == n and token.isalpha():
            words.add(token)
    return words


# Load from each source
rijecalica_4 = load_rijecalica(rijecalica_file, 4)
rijecalica_5 = load_rijecalica(rijecalica_file, 5)
kkrypt0nn_4 = load_set_simple(kkrypt0nn_file, 4)
kkrypt0nn_5 = load_set_simple(kkrypt0nn_file, 5)
hr_4 = load_set_simple(hr_txt_file, 4)
hr_5 = load_set_simple(hr_txt_file, 5)

curated_4_sets = [rijecalica_4, kkrypt0nn_4, hr_4]
curated_5_sets = [rijecalica_5, kkrypt0nn_5, hr_5]

# Strict: word in ALL 3 sources
final_4_strict = rijecalica_4 & kkrypt0nn_4 & hr_4
final_5_strict = rijecalica_5 & kkrypt0nn_5 & hr_5

# Non-strict: word in >=2 sources
count_4 = Counter(w for s in curated_4_sets for w in s)
count_5 = Counter(w for s in curated_5_sets for w in s)
final_4 = {w for w, c in count_4.items() if c >= 2}
final_5 = {w for w, c in count_5.items() if c >= 2}

# Acronym blocklist (extend as needed)
ACRONYM_BLOCKLIST = {
    "adcp", "adsl", "accis", "adac", "adom", "aeif", "afel",
    "adpc", "adns", "aarp", "ascii", "adfs", "adcx", "adcb",
}

def filter_blocklist(words: set[str], blocklist: set[str]) -> set[str]:
    return {w for w in words if w not in blocklist}

final_4 = filter_blocklist(final_4, ACRONYM_BLOCKLIST)
final_5 = filter_blocklist(final_5, ACRONYM_BLOCKLIST)
final_4_strict = filter_blocklist(final_4_strict, ACRONYM_BLOCKLIST)
final_5_strict = filter_blocklist(final_5_strict, ACRONYM_BLOCKLIST)

final_4 = sorted(final_4)
final_5 = sorted(final_5)
final_4_strict = sorted(final_4_strict)
final_5_strict = sorted(final_5_strict)

# Write outputs
out_4.write_text("\n".join(final_4) + ("\n" if final_4 else ""), encoding="utf-8")
out_4_strict.write_text("\n".join(final_4_strict) + ("\n" if final_4_strict else ""), encoding="utf-8")
out_5.write_text("\n".join(final_5) + ("\n" if final_5 else ""), encoding="utf-8")
out_5_strict.write_text("\n".join(final_5_strict) + ("\n" if final_5_strict else ""), encoding="utf-8")

print("Policy summary:")
print("- croatian_4/5_strict: word in ALL 3 sources [Rijecalica, kkrypt0nn, HR_Txt], minus blocklist")
print("- croatian_4/5: word in >=2 sources, minus blocklist")
print()
print("Counts:")
print(f"- croatian_4.txt: {len(final_4)}")
print(f"- croatian_4_strict.txt: {len(final_4_strict)}")
print(f"- croatian_5.txt: {len(final_5)}")
print(f"- croatian_5_strict.txt: {len(final_5_strict)}")
print()
print("Source sizes (4-letter):")
print(f"- rijecalica_4: {len(rijecalica_4)}")
print(f"- kkrypt0nn_4: {len(kkrypt0nn_4)}")
print(f"- hr_txt_4: {len(hr_4)}")
print(f"- added_to_non_strict_4: {len(set(final_4) - set(final_4_strict))}")
print()
print("Source sizes (5-letter):")
print(f"- rijecalica_5: {len(rijecalica_5)}")
print(f"- kkrypt0nn_5: {len(kkrypt0nn_5)}")
print(f"- hr_txt_5: {len(hr_5)}")
print(f"- added_to_non_strict_5: {len(set(final_5) - set(final_5_strict))}")

Policy summary:
- croatian_4/5_strict: word in ALL 3 sources [Rijecalica, kkrypt0nn, HR_Txt], minus blocklist
- croatian_4/5: word in >=2 sources, minus blocklist

Counts:
- croatian_4.txt: 4099
- croatian_4_strict.txt: 2228
- croatian_5.txt: 12228
- croatian_5_strict.txt: 6825

Source sizes (4-letter):
- rijecalica_4: 4470
- kkrypt0nn_4: 2310
- hr_txt_4: 5866
- added_to_non_strict_4: 1871

Source sizes (5-letter):
- rijecalica_5: 13392
- kkrypt0nn_5: 7082
- hr_txt_5: 17716
- added_to_non_strict_5: 5403


## Generate review files (manual cleanup)

Creates `review_croatian_4.txt` and `review_croatian_5.txt` with heuristic-flagged words (candidates for removal).

**Workflow:** Run this cell → open the review files → delete lines for words you want to KEEP → leave only words you want to REMOVE → run the next cell to apply removals.

In [9]:
from pathlib import Path

base = Path("../data")
out_4 = base / "croatian_4.txt"
out_5 = base / "croatian_5.txt"
review_4 = base / "review_croatian_4.txt"
review_5 = base / "review_croatian_5.txt"

words_4 = set(out_4.read_text(encoding="utf-8").splitlines()) if out_4.exists() else set()
words_5 = set(out_5.read_text(encoding="utf-8").splitlines()) if out_5.exists() else set()

VOWELS = set("aeiou")
CROATIAN_VOWEL_LIKE = set("aeioučćđšž")  # diacritics can act as vowels

def vowel_count(w: str) -> int:
    return sum(1 for c in w.lower() if c in VOWELS)

def max_consecutive_consonants(w: str) -> int:
    run, max_run = 0, 0
    for c in w.lower():
        run = run + 1 if c not in CROATIAN_VOWEL_LIKE else 0
        max_run = max(max_run, run)
    return max_run

def has_diacritic(w: str) -> bool:
    return any(c in "čćđšž" for c in w.lower())

def is_all_ascii(w: str) -> bool:
    return all(ord(c) < 128 for c in w)

def flagged_4(w: str) -> list[str]:
    """Return list of reasons this word is flagged for review."""
    reasons = []
    if vowel_count(w) < 1:
        reasons.append("few_vowels")
    if max_consecutive_consonants(w) >= 4:
        reasons.append("many_consonants")
    if is_all_ascii(w) and not has_diacritic(w) and vowel_count(w) <= 1:
        reasons.append("ascii_few_vowels")
    if len(set(w)) <= 2:  # very repetitive
        reasons.append("repetitive")
    return reasons

def flagged_5(w: str) -> list[str]:
    reasons = []
    if vowel_count(w) < 2:
        reasons.append("few_vowels")
    if max_consecutive_consonants(w) >= 4:
        reasons.append("many_consonants")
    if is_all_ascii(w) and not has_diacritic(w) and vowel_count(w) <= 1:
        reasons.append("ascii_few_vowels")
    if len(set(w)) <= 2:
        reasons.append("repetitive")
    return reasons

# Collect flagged words with reasons (for user context)
candidates_4 = []
for w in sorted(words_4):
    reasons = flagged_4(w)
    if reasons:
        candidates_4.append(f"{w}\t# {','.join(reasons)}")

candidates_5 = []
for w in sorted(words_5):
    reasons = flagged_5(w)
    if reasons:
        candidates_5.append(f"{w}\t# {','.join(reasons)}")

review_4.write_text("\n".join(candidates_4) + ("\n" if candidates_4 else ""), encoding="utf-8")
review_5.write_text("\n".join(candidates_5) + ("\n" if candidates_5 else ""), encoding="utf-8")

print(f"review_croatian_4.txt: {len(candidates_4)} candidates (of {len(words_4)} total)")
print(f"review_croatian_5.txt: {len(candidates_5)} candidates (of {len(words_5)} total)")
print()
print("Edit the review files: DELETE lines for words you want to KEEP.")
print("LEAVE only the words you want to REMOVE, then run the next cell.")

review_croatian_4.txt: 607 candidates (of 4099 total)
review_croatian_5.txt: 801 candidates (of 12228 total)

Edit the review files: DELETE lines for words you want to KEEP.
LEAVE only the words you want to REMOVE, then run the next cell.


## Apply manual removals

After you've edited the review files (left only words to remove), run this cell to remove those words from all 4 Croatian datasets.

In [1]:
from pathlib import Path

base = Path("../data")
review_4 = base / "review_croatian_4.txt"
review_5 = base / "review_croatian_5.txt"

def parse_review_file(path: Path) -> set[str]:
    """Extract words to remove (first token per line, before tab or #)."""
    if not path.exists():
        return set()
    words = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        word = line.split("\t")[0].split("#")[0].strip()
        if word:
            words.add(word)
    return words

to_remove_4 = parse_review_file(review_4)
to_remove_5 = parse_review_file(review_5)

print(f"Words to remove: {len(to_remove_4)} from 4-letter, {len(to_remove_5)} from 5-letter")

for path, to_remove in [
    (base / "croatian_4.txt", to_remove_4),
    (base / "croatian_4_strict.txt", to_remove_4),
    (base / "croatian_5.txt", to_remove_5),
    (base / "croatian_5_strict.txt", to_remove_5),
]:
    if not path.exists():
        print(f"Skip (missing): {path.name}")
        continue
    words = [w for w in path.read_text(encoding="utf-8").splitlines() if w and w not in to_remove]
    path.write_text("\n".join(words) + ("\n" if words else ""), encoding="utf-8")
    print(f"Updated {path.name}: {len(words)} words remaining")

Words to remove: 87 from 4-letter, 136 from 5-letter
Updated croatian_4.txt: 4012 words remaining
Updated croatian_4_strict.txt: 2164 words remaining
Updated croatian_5.txt: 12092 words remaining
Updated croatian_5_strict.txt: 6730 words remaining


## Connectivity check (4-letter)

Same as English: build graph, check largest component and average degree.

In [3]:
import networkx as nx
from pathlib import Path
import string

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

# Croatian alphabet includes č, ć, đ, š, ž
CROATIAN_LETTERS = "abcdefghijklmnopqrstuvwxyzčćđšž"

words = set(Path("../data/croatian_4.txt").read_text(encoding="utf-8").splitlines())


def one_letter_neighbors(w: str, letters: str) -> set[str]:
    neighbors = set()
    for i in range(len(w)):
        for c in letters:
            if c != w[i]:
                candidate = w[:i] + c + w[i + 1:]
                if candidate in words:
                    neighbors.add(candidate)
    return neighbors


G = nx.Graph()
G.add_nodes_from(words)
for w in tqdm(words):
    for neighbor in one_letter_neighbors(w, CROATIAN_LETTERS):
        if w < neighbor:
            G.add_edge(w, neighbor)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(G)}")
largest_cc = max(nx.connected_components(G), key=len)
pct = len(largest_cc) / len(words) if words else 0
print(f"Largest component size: {len(largest_cc)} / {len(words)} ({pct:.1%})")

degrees = [d for n, d in G.degree()]
avg_deg = sum(degrees) / len(degrees) if degrees else 0
print(f"Avg degree: {avg_deg:.2f}")

if pct > 0.95:
    print("→ Graph is well-connected; good ladders exist.")
else:
    print("→ Consider revisiting filtering (too strict?).")

100%|██████████| 6315/6315 [00:00<00:00, 23682.23it/s]

Nodes: 6315
Edges: 49352
Connected components: 187
Largest component size: 6093 / 6315 (96.5%)
Avg degree: 15.63
→ Graph is well-connected; good ladders exist.


## Connectivity check (5-letter)

In [ ]:
import networkx as nx
from pathlib import Path

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

CROATIAN_LETTERS = "abcdefghijklmnopqrstuvwxyzčćđšž"
words = set(Path("../data/croatian_5.txt").read_text(encoding="utf-8").splitlines())

def one_letter_neighbors(w: str, letters: str) -> set[str]:
    neighbors = set()
    for i in range(len(w)):
        for c in letters:
            if c != w[i]:
                candidate = w[:i] + c + w[i + 1:]
                if candidate in words:
                    neighbors.add(candidate)
    return neighbors

G = nx.Graph()
G.add_nodes_from(words)
for w in tqdm(words):
    for neighbor in one_letter_neighbors(w, CROATIAN_LETTERS):
        if w < neighbor:
            G.add_edge(w, neighbor)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(G)}")
largest_cc = max(nx.connected_components(G), key=len)
pct = len(largest_cc) / len(words) if words else 0
print(f"Largest component size: {len(largest_cc)} / {len(words)} ({pct:.1%})")
degrees = [d for n, d in G.degree()]
print(f"Avg degree: {sum(degrees)/len(degrees):.2f}" if degrees else "Avg degree: 0")
if pct > 0.95:
    print("→ Graph is well-connected; good ladders exist.")
else:
    print("→ Consider revisiting filtering (too strict?).")